### Import and Setup

In [1]:
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries loaded successfully!")
print(f"XGBoost version: {xgb.__version__}")

Libraries loaded successfully!
XGBoost version: 3.2.0


### Load Data & Optimize Memory

In [4]:
# Load the dataset generated previously
df = pd.read_csv("../addis_traffic_training_data.csv", parse_dates=["timestamp"])

print(f"Original dataset shape: {df.shape}")
print(df.info())

# Memory optimization (downcasting types to keep RAM usage low)
df["hour"] = df["hour"].astype("int8")
df["day_of_week"] = df["day_of_week"].astype("int8")
df["is_weekend"] = df["is_weekend"].astype("int8")
df["length_meters"] = df["length_meters"].astype("float32")
df["max_speed_kmh"] = df["max_speed_kmh"].astype("float32")
df["congestion_index"] = df["congestion_index"].astype("float32")
df["current_speed_kmh"] = df["current_speed_kmh"].astype("float32")

# Sort chronologically just to be certain
df = df.sort_values("timestamp").reset_index(drop=True)
print("Data loaded and optimized for training.")

Original dataset shape: (721000, 10)
<class 'pandas.DataFrame'>
RangeIndex: 721000 entries, 0 to 720999
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   segment_id         721000 non-null  int64         
 1   timestamp          721000 non-null  datetime64[us]
 2   hour               721000 non-null  int64         
 3   day_of_week        721000 non-null  int64         
 4   is_weekend         721000 non-null  int64         
 5   length_meters      721000 non-null  float64       
 6   max_speed_kmh      2163 non-null    float64       
 7   congestion_index   721000 non-null  float64       
 8   current_speed_kmh  2163 non-null    float64       
 9   traffic_status     721000 non-null  str           
dtypes: datetime64[us](1), float64(4), int64(4), str(1)
memory usage: 55.0 MB
None
Data loaded and optimized for training.


### Feature Engineering and Feature Selection

In [5]:
# Define features (X) and target (y)
# We will predict 'congestion_index' (0.0 to 1.0)
feature_cols = [
    "hour",
    "day_of_week",
    "is_weekend",
    "length_meters",
    "max_speed_kmh",
    "segment_id",
]

X = df[feature_cols]
y = df["congestion_index"]

print(f"Features selected: {feature_cols}")
print(f"Target variable: congestion_index")

Features selected: ['hour', 'day_of_week', 'is_weekend', 'length_meters', 'max_speed_kmh', 'segment_id']
Target variable: congestion_index


### Chronological Train/Test split

In [6]:
# Split 80% for training, 20% for testing chronologically
split_index = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (576800, 6)
Testing set shape: (144200, 6)


### Train XGBoost Regressor

In [7]:
# Initialize XGBoost Regressor with optimal parameters for speed and performance
model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,  # Uses all CPU cores
)

print("Training XGBoost model...")
model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=10,  # Prints progress every 10 trees
)
print("Training completed!")

Training XGBoost model...
[0]	validation_0-rmse:0.29066
[10]	validation_0-rmse:0.12889
[20]	validation_0-rmse:0.08634
[30]	validation_0-rmse:0.07337
[40]	validation_0-rmse:0.07134
[50]	validation_0-rmse:0.07080
[60]	validation_0-rmse:0.07065
[70]	validation_0-rmse:0.07061
[80]	validation_0-rmse:0.07061
[90]	validation_0-rmse:0.07062
[99]	validation_0-rmse:0.07062
Training completed!


### Evaluate model performance

In [8]:
# Predict on the test set
y_pred = model.predict(X_test)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Model Evaluation Metrics ---")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-squared (R2 Score): {r2:.4f}")

--- Model Evaluation Metrics ---
Root Mean Squared Error (RMSE): 0.0706
Mean Absolute Error (MAE): 0.0586
R-squared (R2 Score): 0.9423


### Save model artifact

In [9]:
# Save the trained model using joblib (standard for scikit-learn compatible models)
model_filename = "addis_traffic_xgboost_model.pkl"
joblib.dump(model, model_filename)

print(f"Model successfully saved to '{model_filename}'!")

Model successfully saved to 'addis_traffic_xgboost_model.pkl'!


In [10]:
print("1. Loading the saved XGBoost model...")
model = joblib.load("addis_traffic_xgboost_model.pkl")

# 2. Define real-life input data
# Example: Simulating a request for a specific road segment during Monday morning rush hour (8:00 AM)
incoming_request_data = {
    "hour": [8],
    "day_of_week": [0],  # Monday
    "is_weekend": [0],  # Weekday
    "length_meters": [150.5],  # Length of the road segment in meters
    "max_speed_kmh": [50.0],  # Speed limit
    "segment_id": [12345678],  # OpenStreetMap segment ID
}

# Convert input dictionary into a pandas DataFrame (matching training features)
input_df = pd.DataFrame(incoming_request_data)

print("3. Sending input to the model for prediction...")
predicted_congestion = model.predict(input_df)[0]  # Get the float value


# 4. Post-process the output into human-readable traffic status
def get_status_label(congestion_val):
    if congestion_val < 0.3:
        return "Smooth"
    elif congestion_val < 0.7:
        return "Moderate"
    else:
        return "Heavy"


status = get_status_label(predicted_congestion)

print("\n--- Real-Life Prediction Result ---")
print(f"Predicted Congestion Index: {predicted_congestion:.4f}")
print(f"Traffic Status: {status}")

1. Loading the saved XGBoost model...
3. Sending input to the model for prediction...

--- Real-Life Prediction Result ---
Predicted Congestion Index: 0.8465
Traffic Status: Heavy
